# FORCE Level 2 processing using openEO on CDSE with pystac Query

### Purpose

The [Earth Observation Application Package (EOAP)](https://docs.ogc.org/bp/20-089r1.html) and surrounding tools implemented in the [`apex-force-openeo`](https://github.com/bcdev/apex-force-openeo) repository make it possible to use the [FORCE](https://force-eo.readthedocs.io/en/latest/) EO processing engine on the [Copernicus Data Space Ecosystem (CDSE)](https://dataspace.copernicus.eu/) using the [openEO](https://openeo.dataspace.copernicus.eu/) API.
This notebook uses `pystac_client` to make a STAC query to determine input products for processing with FORCE on the CDSE OpenEO backend.

> This notebook serves to showcase a simple workflow to run FORCE level 2 on CDSE

### Prerequisites

In order to run this notebook, you need to have a CDSE account and sufficient openEO credits to run the jobs. Please note that CDSE provides a contingent of free credits each month, so you should be able to run the notebook if you haven't used up your credits, with no actual cost.

### Content

1. Connect to OpenEO backend
2. Make a query to the CDSE STAC catalog to determine input products
3. Run the FORCE l2ps on CDSE using OpenEO

### Citation and Acknowledgement

When using the FORCE processing engine, please make sure to properly [cite and acknowledge](https://force-eo.readthedocs.io/en/latest/policy/citation.html)
the software and its related [scientific publications](https://force-eo.readthedocs.io/en/latest/refs.html#refs).
The implementation as EOAP was performed in the context of ESA's [APEx](https://apex.esa.int/) initiative.

### Further reading

**FORCE on CDSE using OpenEO**

1. [User guide](https://esa-apex.github.io/apex_toolbox_documentation/docs/force/guide/intro.html)
2. [Parameters](https://esa-apex.github.io/apex_toolbox_documentation/docs/force/guide/parametrization.html)

**APEx**

1. [Project website](https://apex.esa.int/)
2. [Documentation Portal](https://esa-apex.github.io/apex_documentation/)

**FORCE**

1. [Documentation](https://force-eo.readthedocs.io/en/latest/index.html)
1. [Level 2 processing system](https://force-eo.readthedocs.io/en/latest/components/lower-level/level2/)
1. [Citation and acknowledgement](https://force-eo.readthedocs.io/en/latest/policy/citation.html)


**openEO**

1. [openEO on CDSE](https://documentation.dataspace.copernicus.eu/APIs/openEO/openEO.html)
1. [Credit Usage on CDSE](https://documentation.dataspace.copernicus.eu/APIs/openEO/credit_usage.html)
1. [openEO project website](https://openeo.org/)
1. [openEO Python Client documentation](https://open-eo.github.io/openeo-python-client/)


### Imports

In [ ]:
from pathlib import Path
import getpass

import openeo
import contextily
import shapely
from shapely import Polygon
import pystac
import pystac_client
import matplotlib.pyplot as plt
import time
import geopandas as gpd
import contextily as ctx

### Area of Interest

For this example, we selected a small area around Modena on Italy's Adriatic Sea coast. 
Here, we quickly visualize the area of interest.

In [ ]:
w,s,e,n = 10.386, 44.437, 11.423, 44.973
spatial_extent = { "west": w, "south": s, "east": e, "north": n}
aoi_name = "Modena"
temporal_extent=["2026-04-17", "2026-04-18"]

In [ ]:
def plot_area_of_interest(
    w, s, e, n, *,
    large_context=10,
    small_context=1.25,
    color="red",
    linewidth=2,
    title="Spatial Extent",
    figsize=(12, 8),
    show_tick_labels=True,
    basemap_source=ctx.providers.OpenTopoMap,
    ):
    polygon = Polygon([(w, s), (e, s), (e, n), (w, n)])
    gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
    fig, axs = plt.subplots(1, 2, figsize=figsize)
    gdf.plot(edgecolor=color, facecolor="none", linewidth=linewidth, ax=axs[0])
    gdf.plot(edgecolor=color, facecolor="none", linewidth=linewidth, ax=axs[1])
    axs[0].set_xlim(w - large_context, e + large_context)
    axs[0].set_ylim(s - large_context, n + large_context)
    axs[1].set_xlim(w - small_context, e + small_context)
    axs[1].set_ylim(s - small_context, n + small_context)
    if not show_tick_labels:
        for ax in axs:
            ax.set(xticks=[], yticks=[])
    ctx.add_basemap(axs[0], source=basemap_source, crs=gdf.crs)
    ctx.add_basemap(axs[1], source=basemap_source, crs=gdf.crs)
    fig.suptitle(title)
    fig.tight_layout()

In [ ]:
plot_area_of_interest(
    w=w,
    s=s,
    e=e,
    n=n,
    large_context=4,
    figsize=(10, 6),
    title=f"Spatial Extent: {aoi_name}"
)


### Connect to the OpenEO backend

In order to send requests and start jobs on the openEO backend, we must first authenticate. If you haven't recently logged into the openEO backend, this cell will prompt you to log in. Please click the link produced in the cell to log in.

> If you don't have a CDSE account yet, please [create one first](https://documentation.dataspace.copernicus.eu/Registration.html).

In [ ]:
backend_url = "openeo.dataspace.copernicus.eu"
connection = openeo.connect(backend_url).authenticate_oidc()

### Query the CDSE STAC catalog

To determine the inputs to process with FORCE, we need to determine a number of STAC Items, describing our input products. You can pass any `pystac.ItemCollection`, `pystac.Item`, `pystac.Collection` or `pystac.Catalog` to the process. Make sure that total input data is of reasonable size (e.g, avoid passing a STAC catalog of the Sentinel-2-L1C archive).

The easiest way to get an `ItemCollection` is to use `pystac_client.Client` and do a search:

In [ ]:
L1C_COLLECTION_URL = "https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l1c"

client = pystac_client.Client.open("https://stac.dataspace.copernicus.eu/v1")
search = client.search(
    datetime=temporal_extent,
    max_items=6,
    collections=["sentinel-2-l1c"],
    bbox=[w, s, e, n]
)
item_collection = search.item_collection()

print("Found items:")
for item in item_collection.items:
    time.sleep(0.2) # CDSE STAC API has a rate limit of 5 requests / s
    print(item.id)


### Running FORCE level 2 (l2ps)

Now that we have selected the input files, we can pass the selection to the FORCE level 2 process. The process will automatically download the necessary files to the executor on the CDSE infrastructure.

#### Observing the progress on openEO

You can track the progress of your job by logging into the [openEO web interface](https://openeo.dataspace.copernicus.eu/).

Alternatively, in a separate Python shell / notebook, you can run the following snippet to get access to the logs as a Python list.
You can find out the job id from the next cell below.

Please be aware that the output from CWL jobs is not optimally rendered by openEO logs, so the logs may look a little bit messy and fragmented. However, they can be a valuable source of information to debug issues.

```Python
JOB_ID = ... # see the output from `l2_job.start_and_wait()` below)
connection = ... # connect to the same backend
l2_job = connection.job(JOB_ID)
l2_logs = l2_job.logs()
print("\n".join(l2_logs[:20])) # show the first 20 logs.
l2_logs # openEO logs have rich rendering and a searchable interface in jupyter notebooks
```

#### How does the openEO code work?

First, we create a [StacResource](https://open-eo.github.io/openeo-python-client/api.html#openeo.rest.stac_resource.StacResource) which allows us to handle a data cube as a STAC object (and not an openEO data cube). This is necessary, because FORCE does not use the standard  openEO data cube format.

The STAC resource takes a [PGNode](https://open-eo.github.io/openeo-python-client/api.html#openeo.internal.graph_building.PGNode) (a process graph node) specifying the process used to create the `StacResource`. The process we use is a custom process for FORCE level2.

> Once the FORCE custom process is available, it will no longer be necessary to specify the CWL document, as the process will know which one to use. For now, we have to use the generic `run_cwl_to_stac` process.
We create a StacResource

#### Parametrization

Most parameters of FORCE level 2 are supported and can be passed to the `arguments` parameter of the `force_level2` process.

Note that the parameters have the same name as in the FORCE parameter file, but in `lower_snake_case` to conform with openEO convention. 

> The EOAP supports most parameters that can be specified in the parameter file with a few exceptions:
> - Parameters that set input/output directories, these are managed by the EOAP
> - Parameters that specify processing details, e.g., `NPROC`
> - Only the `Copernicus_30m` digital elevation model is supported (parameter `DEM`/`dem`)
>
> For more details have a look at the [feature overview](https://esa-apex.github.io/apex_toolbox_documentation/docs/force/feature_overview.html) and [parametrization](https://esa-apex.github.io/apex_toolbox_documentation/docs/force/guide/parametrization.html) sections in the documentation.


#### Further processing

This example allows further processing only after download of the results to your local machine. To continue processing with FORCE TSA, you need to add an [`export_workspace`](https://open-eo.github.io/openeo-python-client/api.html#openeo.rest.stac_resource.StacResource.export_workspace) node to your process graph as described in the [TSA example](../run-level2-and-tsa/FORCE_level2_and_TSA.ipynb) and the [documentation](https://esa-apex.github.io/apex_toolbox_documentation/docs/force/guide/level2.html#workspaces).


In [ ]:
def transform_item_collection_to_catalog_with_links(item_collection):
    catalog = pystac.Catalog(
        id="item-collection-catalog",
        description="Catalog from item collection",
    )
    for item in item_collection.items:
        item_link = pystac.Link(
            rel=pystac.RelType.ITEM,
            target=item.self_href,
            media_type=pystac.MediaType.GEOJSON,
            title=f"Item {item.id}",
        )
        catalog.add_link(item_link)
    return catalog

query_catalog = transform_item_collection_to_catalog_with_links(item_collection)

### Saving level 2 results in a workspace


When running FORCE level2 on its own, we do not need to use a workspace. Simply download the results when you are done with processing. However, to reference the level 2 processing results in further processing, we need to use a workspace.

OpenEO uses the concept of Workspaces to interface with object storage. For processing with FORCE, we can use a workspace dedicated to FORCE to store and exchange processing results between level 2 and higher level processing, and with colleagues. We can use either the dedicated workspace apex-force-results-workspace, which is accessible to all CDSE openeo users, or another registered workspace. See the openEO platform documentation on workspaces on how workspaces can be accessed and registered.

We can instruct openEO to store our results in a workspace using the export_workspace openEO process. When using export_workspace, we need to pass a merge string to identify your result. The merge will be a key prefix on object storage. The merge is something similar to a path in a file system. It is very important to set a good merge parameter and remember the result, because results can only be accessed when the merge is known. Furthermore, all users of the workspace (in the case of the apex-force-results-workspace, anyone) may write data, so make sure to use a unique merge, to reduce the likelihood of accidentally modifying others’ data and have others overwrite your data by accident.


The dedicated workspace for FORCE results is accessible to all users:

    id: apex-force-results-workspace
    url: https://s3.waw4-1.cloudferro.com/apex-force-results-waw4-1-exotc5yuexi2c5tvwqhoivj62fz8v0uupy0me



In [ ]:
processing_name = "FORCE_level2_modena"

user = getpass.getuser() # running on your machine? Your username is used as merge. It may not be unique, so better choose a unique name.
if user == "web_user": # running via jupyterlite? Please set the merge by hand!
    user = None
    # merge = "unique-string"  # <--- set our own merge path
    
# Be careful: The merge determines the storage location of your results on the publicly accessible apex-force-results-workspace.
# You may overwrite your results or those of others, if the `merge` parameter is not
# Set to a unique value.
if not merge and user is not None:
    merge = f"{user}/experiments/{processing_name}"
    

In [ ]:
from openeo.rest.stac_resource import StacResource
from openeo.internal.graph_building import PGNode

args = {
    "stac_document": item_collection.to_dict(),
    "name": processing_name,
    "do_brdf": True,
    "output_ovv": True,
    # <----- more parameters
}

force_l2_stac_resource = StacResource(
    graph=PGNode(
        process_id="force_level2",
        arguments={
            "stac_document": query_catalog.to_dict(),
            "name": processing_name,
            "do_brdf": True,
        }
    ),
    connection=connection,
)

# Optional when only processing level 2
# If you intend to continue processing, or share the results with others,
# add the `export_workspace` call:
# Make sure merge is set

if not merge:
    raise RuntimeError("Please specify a merge path (cell above)")
    
print("saving results to workspace with merge: ", merge)
force_l2_stac_resource = force_l2_stac_resource.export_workspace(
    workspace="apex-force-results-workspace",
    merge=merge,
)
l2_job = force_l2_stac_resource.create_job(title=f"FORCE level 2 - {aoi_name}: {merge}", job_options = {"export-workspace-enable-merge": False})
l2_job.start_and_wait()

## Result access

If you included the `export_workspace` step above in the process graph, the results are publicly availble on the apex-force-results workspace, you may use any S3 tools to download them. For example, use [`s5cmd`](https://github.com/peak/s5cmd):

```bash
# Downloads the FORCE data cube to the current working directory:
# Replace <merge> with your merge path.
esttest/europe
s5cmd --endpoint-url https://s3.waw4-1.cloudferro.com/ --no-sign-request cp s3://apex-force-results-waw4-1-exotc5yuexi2c5tvwqhoivj62fz8v0uupy0me/<merge>/* .
```

Without export workspace, you can use openEO's standard result access tools (`l2_results.download_files(<path>)`).

Alternatively, you can continue processing with the [FORCE Time Series Analysis](https://algorithm-catalogue.apex.esa.int/apps/force_tsa) process (requires usage of `export_workspace`).

For more and longer examples, see https://github.com/bcdev/apex-force-openeo/tree/main/examples


In [ ]:
l2_results = l2_job.get_results()
l2_results